In [217]:
import random
import numpy as np
from activators import SigmoidActivator,IdentityActivator

In [218]:
class FullConnectedLayer(object):
    def __init__(self, input_size, output_size,activator):
        self.input_size = input_size
        self.output_size = output_size
        self.activator = activator
        # 权重数组 W
        # 形状：(output_size, input_size)
        # 为什么用这个形状？因为计算时：output = W · input + b
        # 如果input是(n,1)列向量，W是(m,n)，那么W·input是(m,1)
        self.W =np.random.uniform(-0.1,0.1,(output_size,input_size))
        self.b=np.zeros((output_size,1))
        self.output=np.zeros((output_size,1))

        # 反向传播
        self.input=None
        self.delta=None
        self.W_grad=None
        self.b_grad=None

    def forward(self,input_array):
        self.input=input_array
        # 计算加权输入：net = W·input + b
        net=np.dot(self.W,input_array)+self.b
        self.output=self.activator.forward(net)
    def backward(self,delta_array):
        # delta_array = δ^l
        # 这里计算对上一层的误差
        self.delta=self.activator.backward(self.input)*np.dot(self.W.T,delta_array)
        # 计算权重的梯度：∂E/∂W^l = δ^l · (a^{l-1})^T
        # 其中 a^{l-1} = input_array（本层的输入）
        self.W_grad=np.dot(delta_array,self.input.T)
        self.b_grad=delta_array
    def update(self,learning_rate):
        self.W+=learning_rate * self.W_grad
        self.b+=learning_rate * self.b_grad


In [219]:
class Network(object):
    def __init__(self,layers):
        self.layers=[]
        # 创建输入层到输出层之间的所有层，不包括输入层
        for i in range(len(layers)-1):
            self.layers.append(FullConnectedLayer(layers[i],layers[i+1],SigmoidActivator()))
    def predict(self,sample):
        output=sample
        for layer in self.layers:
            layer.forward(output)
            output = layer.output
        return output
    def train(self,labels,data_set,rate,epoch):
        for epoch in range(epoch):
            for d in range(len(data_set)):
                self.train_one_sample(labels[d],data_set[d],rate)
    def train_one_sample(self,label,sample,rate):
        self.predict(sample)
        self.calc_gradient(label)
        self.update_weight(rate)
    def calc_gradient(self,label):
        delta=self.layers[-1].activator.backward(self.layers[-1].output)*(label-self.layers[-1].output)
        for layer in self.layers[::-1]:
            layer.backward(delta)
            delta=layer.delta
        return delta
    def update_weight(self,rate):
        for layer in self.layers:
            layer.update(rate)
    def loss(self,output,label):
        return 0.5 *((label-output)*(label-output)).sum()




In [220]:
from functools import reduce


class Normalizer(object):
    def __init__(self):
        self.mask = [
            0x1, 0x2, 0x4, 0x8, 0x10, 0x20, 0x40, 0x80
        ]

    def norm(self, number):
        # 使用列表推导式，Python 2和3都兼容
        data = [0.9 if number & m else 0.1 for m in self.mask]
        return np.array(data).reshape(8, 1)

    def denorm(self, vec):
        # 同样修改这里
        binary = [1 if i > 0.5 else 0 for i in vec[:, 0]]
        for i in range(len(self.mask)):
            binary[i] = binary[i] * self.mask[i]
        return sum(binary)  # reduce(lambda x,y: x + y, binary) 改为 sum()

In [221]:
def train_data_set():
    normalizer = Normalizer()
    data_set = []
    labels = []
    for i in range(0, 256):
        n = normalizer.norm(i)  # 将数字i编码成8维向量
        data_set.append(n)      # 输入
        labels.append(n)        # 输出（和输入相同）
    return labels, data_set
def correct_ratio(network):
    '''计算网络的正确率（对于自编码器，检查是否能重建原始数字）'''
    normalizer = Normalizer()
    correct = 0.0
    for i in range(256):
        # 输入数字i，得到输出，再解码回数字
        if normalizer.denorm(network.predict(normalizer.norm(i))) == i:
            correct += 1.0
    print('correct_ratio: %.2f%%' % (correct / 256 * 100))

In [222]:
def test():
    labels, data_set = train_data_set()


    net = Network([8, 3, 8])

    rate = 0.5
    epoch = 16

    for i in range(epoch):
        net.train(labels, data_set, rate, epoch)
        print('after epoch %d loss: %f' % (
            (i + 1),
            net.loss(labels[-1], net.predict(data_set[-1]))
        ))
        rate /= 2       # 每轮降低学习率

    correct_ratio(net)


if __name__ == '__main__':
    test()

after epoch 1 loss: 0.052916
after epoch 2 loss: 0.079414
after epoch 3 loss: 0.124666
after epoch 4 loss: 0.181173
after epoch 5 loss: 0.231950
after epoch 6 loss: 0.265845
after epoch 7 loss: 0.284502
after epoch 8 loss: 0.293382
after epoch 9 loss: 0.297523
after epoch 10 loss: 0.299512
after epoch 11 loss: 0.300487
after epoch 12 loss: 0.300970
after epoch 13 loss: 0.301210
after epoch 14 loss: 0.301330
after epoch 15 loss: 0.301390
after epoch 16 loss: 0.301420
correct_ratio: 7.42%
